# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/T0othIess/FlyRank-AI-ML-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [22]:
import pandas as pd
import os
from  huggingface_hub import login
import duckdb

HF_TOKEN = os.environ.get("HF_TOKEN")
login(HF_TOKEN)
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
rel = "hf://datasets/FlyRank/internship-warehouse"
fact_content_daily_performance_table = con.sql(f"SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')")
dim_content_table  = con.sql(f"SELECT * from read_parquet('{rel}/dim_content.parquet')")

df = con.sql("""SELECT f.content_hash_id, f.gsc_clicks, f.gsc_impressions, f.gsc_avg_position, d.search_volume, d.competition_level, d.main_intent
             FROM fact_content_daily_performance_table AS f JOIN dim_content_table AS d USING(content_hash_id)
             WHERE f.gsc_data_available IS TRUE AND d.is_deleted IS FALSE AND f.gsc_avg_position > 0""").df()

print("Percentages of pages with NaN in each column:")
#TO explain this .map() logic, first of all, template.format works like, apply template to whats in format function, format comes in python not a panda fucntion,
#the reason format doesnt have () is because .map works on each row of the left side, so for each row it does like this: "{:.2f}%".format(row), which applies the template to the row.
#what coulve worked aswell is using lambdas as such: .map(lambda row: "{:.2f}%".format(row))
print(df[["search_volume", "competition_level", "main_intent"]].isna().multiply(100).mean().map("{:.2f}%".format))

print(f"OLD DATAFRAME LENGTH: {len(df)}")
#since the pcts we got are very low, we can delete them from the df
df = df.dropna(subset=["search_volume", "competition_level", "main_intent"])
print(f"NEW DATAFRAME LENGTH: {len(df)}")

#unordered category type, just to make main_intent column actual categories not just strings
df["main_intent"] = df["main_intent"].astype("category")

#this is ordered category, self explanatory.
category_ranks = pd.api.types.CategoricalDtype(categories=["LOW", "MEDIUM", "HIGH"], ordered=True)
df["competition_level"] = df["competition_level"].astype(category_ranks)

#reminder this returns a panda type called category
df["position_tier"] = pd.cut(df["gsc_avg_position"], bins=[0,10,20,float("inf")], labels=["page_1", "striking", "page_3_5"])
expected_ctr_per_tier = df.groupby("position_tier")["gsc_clicks"].sum() / df.groupby("position_tier")["gsc_impressions"].sum()
df["expected_ctr"] = df["position_tier"].map(expected_ctr_per_tier).astype(float)
df["ctr"] = df["gsc_clicks"] / df["gsc_impressions"].mask(df["gsc_impressions"] == 0)
df["ctr_gap"] = df["expected_ctr"] - df["ctr"]

#this removes the same page id duplicates so that the top K rows dont have same ids (each content is per day so it could happen)
df_best = df.sort_values(["ctr_gap", "search_volume", "competition_level"], ascending=[False,False,False]).drop_duplicates(subset=["content_hash_id"], keep="first")
df_best.head(50).style.format("{:.3f}", subset=["expected_ctr", "ctr", "ctr_gap"]).set_properties(**{"text-align": "center"})

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Percentages of pages with NaN in each column:
search_volume        3.13%
competition_level    3.54%
main_intent          2.93%
dtype: str
OLD DATAFRAME LENGTH: 3446103
NEW DATAFRAME LENGTH: 3324056


,content_hash_id,gsc_clicks,gsc_impressions,gsc_avg_position,search_volume,competition_level,main_intent,position_tier,expected_ctr,ctr,ctr_gap
205871,content_04e4047dc8eef2fd,0,1,10.000000,368000,HIGH,transactional,page_1,0.003,0.000,0.003
420390,content_e88504a4c6d64b79,0,23,5.956522,201000,HIGH,commercial,page_1,0.003,0.000,0.003
1424873,content_621e4dc78b849ce4,0,2,5.000000,201000,LOW,informational,page_1,0.003,0.000,0.003
2286201,content_a31c400b511b1458,0,5,9.000000,201000,LOW,informational,page_1,0.003,0.000,0.003
2018098,content_3d0d560b0853ce2b,0,1,6.000000,135000,HIGH,transactional,page_1,0.003,0.000,0.003
2863840,content_ff42f4a65f10744c,0,1,10.000000,135000,LOW,informational,page_1,0.003,0.000,0.003
392448,content_e7e56b5396c93880,0,25,9.760000,110000,LOW,informational,page_1,0.003,0.000,0.003
478591,content_6011e836cf18643a,0,2,4.000000,110000,LOW,informational,page_1,0.003,0.000,0.003
1318959,content_9755ef5214465568,0,18,4.555556,110000,LOW,informational,page_1,0.003,0.000,0.003
1291874,content_699272cf5c13d534,0,1,2.000000,90500,MEDIUM,informational,page_1,0.003,0.000,0.003


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.